# Disease Burden Metrics Calculation

**User Story 2**: Calculate Multi-Dimensional Burden Metrics

**Objective**: Calculate comprehensive burden metrics across multiple dimensions (volume, trends, outbreaks, variability) for all 44 diseases to enable evidence-based disease prioritization.

**Data Period**: 2012-2020 (470 weeks)

**Metrics Calculated**:
1. **Volume Metrics**: Total cases, annual average, peak weekly, incidence rate
2. **Trend Metrics**: Linear trend, CAGR, Mann-Kendall test, trend direction
3. **Outbreak Metrics**: Threshold, frequency, duration, intensity
4. **Variability Metrics**: CV, IQR, volatility score
5. **Composite Burden Score**: Weighted combination of normalized metrics

## 1. Setup and Import Libraries

In [1]:
# Standard library imports
import sys
import logging
from pathlib import Path
from datetime import datetime

# Third-party imports
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add src directory to path
sys.path.append(str(Path.cwd().parent.parent))

# Import custom modules
from src.config import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RESULTS_TABLES_DIR,
    RESULTS_FIGURES_DIR,
    BURDEN_METRICS_PATH,
    SINGAPORE_POPULATION
)
from src.data_processing.burden_metrics import (
    calculate_volume_metrics,
    calculate_variability_metrics,
    normalize_metrics,
    calculate_composite_burden_score,
    flag_data_quality
)
from src.analysis.trend_analysis import calculate_all_trend_metrics
from src.analysis.outbreak_detection import calculate_all_outbreak_metrics

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully")
print(f"Python version: {sys.version}")
print(f"Polars version: {pl.__version__}")
print(f"NumPy version: {np.__version__}")

✅ All libraries imported successfully
Python version: 3.9.6 (default, Dec  2 2025, 07:27:59) 
[Clang 17.0.0 (clang-1700.6.3.2)]
Polars version: 0.19.0
NumPy version: 1.26.2


## 2. Load Cleaned Disease Data

In [3]:
# Load cleaned disease data from User Story 1
# Use absolute path relative to project root
data_path = Path("../../data/3_interim/cleaned_disease_data.parquet")

logger.info(f"Loading data from {data_path}")
df = pl.read_parquet(data_path)

# Display data summary
print(f"✅ Data loaded successfully")
print(f"   Rows: {df.height:,}")
print(f"   Columns: {df.width}")
print(f"   Diseases: {df['disease_name'].n_unique()}")
print(f"   Time period: {df['year'].min()} - {df['year'].max()}")
print(f"   Weeks: {df['epidemiological_week'].n_unique()}")

# Display first few rows
df.head()

2026-02-11 11:57:31,698 - __main__ - INFO - Loading data from ../../data/3_interim/cleaned_disease_data.parquet


✅ Data loaded successfully
   Rows: 16,066
   Columns: 10
   Diseases: 44
   Time period: 2012 - 2020
   Weeks: 470


epidemiological_week,year,week,week_start_date,week_end_date,disease_name,case_count,is_outlier,transmission_mode,burden_tier
str,i32,i32,date,date,str,i64,bool,str,str
"""2012-W01""",2012,1,2012-01-02,2012-01-08,"""Acute Viral he…",0,false,"""Other""","""High"""
"""2012-W01""",2012,1,2012-01-02,2012-01-08,"""Acute Viral he…",0,false,"""Other""","""Medium"""
"""2012-W01""",2012,1,2012-01-02,2012-01-08,"""Avian Influenz…",0,false,"""Other""","""Low"""
"""2012-W01""",2012,1,2012-01-02,2012-01-08,"""Campylobactere…",6,false,"""Other""","""High"""
"""2012-W01""",2012,1,2012-01-02,2012-01-08,"""Chikungunya Fe…",0,false,"""Vector-borne""","""High"""
